# Federate learning without differential privacy

This notebook runs the federated learning (FL) baseline for the SoK experiments on accuracy, i.e., without differential privacy (DP)
We simulate FL settings using the private federated learning library ([PFL](https://apple.github.io/pfl-research/)).

In [ ]:
import torch
import sys
import os
import numpy as np
import types
import pandas as pd
import json

import nest_asyncio
nest_asyncio.apply()

from pfl.aggregate.simulate import SimulatedBackend
from pfl.algorithm import FederatedAveraging, NNAlgorithmParams
from pfl.callback import CentralEvaluationCallback, AggregateMetricsToDisk
from pfl.hyperparam import NNTrainHyperParams, NNEvalHyperParams
from pfl.data.dataset import Dataset
from pfl.model.pytorch import PyTorchModel



device = "cuda" if torch.cuda.is_available() else "cpu"
# If on Apple Silicon, use MPS
# device = "mps"
# os.environ['PFL_PYTORCH_DEVICE'] = device


sys.path.append('./dataset/')
from dataset.fashion_mnist.load_preprocess import load_and_preprocess_fashion_mnist_federated
from dataset.mnist.load_preprocess import load_and_preprocess_mnist_federated

sys.path.append('./utils/')
from utils.models import ThreeLayerNN
from utils.mpc_dpsgd_trainer import DP_Trainer
from utils.metrics import image_classification_loss, image_classification_metrics
from utils.pfl_logging_parser import save_best_stats_per_iteration

## Model
The model used for the experiments is a 3 layers neural network. Here, the model is adapted to the federated settings via the `get_pfl_model` function.

In [ ]:
base_folder_metrics = "dl_non_dp_training"
base_path_metrics = f"{base_folder_metrics}/metrics"

def get_pfl_model():
	model = ThreeLayerNN(
		input_size=784,
		hidden_size=100,
		output_size=10
	)


	model.loss = types.MethodType(image_classification_loss,
									model)
	model.metrics = types.MethodType(image_classification_metrics,
										model)
	print(f'PFL model: \n{model}')
	pfl_params = [p for p in model.parameters() if p.requires_grad]


	# Since we rely on FederatedAveraging, which averages the model updates from clients,
	# we set the central learning rate to 1.0 to avoid scaling the updates again.
	central_learning_rate = 1.0

	pfl_model = PyTorchModel(model=model,
				local_optimizer_create=torch.optim.SGD,
				central_optimizer=torch.optim.SGD(pfl_params, central_learning_rate))
	
	return pfl_model

## Dataset
The experiments can be run on either MNIST or Fashion-MNIST datasets by changing the `dataset` variable below.

In [ ]:
dataset_name = "mnist"
#dataset_name = "fashion_mnist"

## Training

### Hyperparameter setting


In [ ]:
# The number of central iterations
central_iterations = 1000
local_iteration = 5
# MNIST and Fashion-MNIST datasets have 60000 training samples
mnist_sample_size = 60000
# The population size is the number of total users in the federated setting
population_size = 1000
# The cohort size is the number of users selected in each round
cohort_size = 100
samples_per_user = mnist_sample_size // population_size
#samples_per_user = 60

learning_rate = 0.1
epochs = 10
momentum = 0 # 0.9
weight_decay = 0 # 0.0001

print(f"Samples per user: {samples_per_user}")

local_batch_size =  10 if samples_per_user >= 10 else samples_per_user
sampling_probability = (cohort_size * samples_per_user) / mnist_sample_size

### Training loop
Here, we define the training loop which first converts the dataset into a federated variant using utilities from the `pfl` library (More details in `dataset/mnist/load_and_preprocess_mnist_federated.py` and `dataset/fashion_mnist/load_and_preprocess_fashion_mnist_federated.py`). Then, we define the federated learning network and the local code that each client and the central server have to execute. We selected as learning algorithm Federated Averaging (FedAvg).

In [ ]:
def non_dp_training_run(
	pfl_model,	
	dataset_name,
	cohort_size,
	local_iterations, 
	local_batch_size,
	learning_rate, 
	samples_per_user,
	central_iterations,
	cnt_iteration, 
):
	if dataset_name == "mnist":
		train_data_federated, val_data_federated, val_data_central = load_and_preprocess_mnist_federated(samples_per_user=samples_per_user, normalization=True, scaling=True)
	elif dataset_name == "fashion_mnist":
		train_data_federated, val_data_federated, val_data_central = load_and_preprocess_fashion_mnist_federated(samples_per_user=samples_per_user, normalization=True, scaling=True)
	else:
		raise ValueError(f"Unsupported dataset: {dataset_name}")

	evaluation_frequency = 2
	
	algorithm_params = NNAlgorithmParams(
      central_num_iterations=central_iterations,
      evaluation_frequency=evaluation_frequency,
      train_cohort_size=cohort_size,
      val_cohort_size=100)
	
	model_train_params = NNTrainHyperParams(
      local_num_epochs=local_iterations,
      local_learning_rate=learning_rate,
      local_batch_size=local_batch_size
	  )

	# Do full-batch evaluation to run faster.
	model_eval_params = NNEvalHyperParams(local_batch_size=None)

	backend = SimulatedBackend(training_data=train_data_federated,
                           val_data=val_data_federated,
                           postprocessors=[])

	central_data = Dataset(raw_data=[val_data_central.data, val_data_central.targets])
	pfl_callbacks = [CentralEvaluationCallback(central_data, model_eval_params, evaluation_frequency), AggregateMetricsToDisk(output_path=f"{base_path_metrics}_{cnt_iteration}.csv")]

	algorithm = FederatedAveraging()

	pfl_model = algorithm.run(
		backend=backend,
		model=pfl_model,
		algorithm_params=algorithm_params,
		model_train_params=model_train_params,
		model_eval_params=model_eval_params,
		callbacks=pfl_callbacks,
		send_metrics_to_platform=True
	)

### Run training
Note that the `cnt_iteration` variable is used to save the metrics file and json file with different names for each run, and can be used to run different experiment in sequence or a hyperparameter search.

In [ ]:
torch.random.manual_seed(0)
np.random.seed(0)


pfl_model = get_pfl_model()
# The cnt_iteration variable is used to save the metrics file and json file with different names for each run
# This variable can be used in a hyperparameter tuning loop
cnt_iteration = 0 

non_dp_training_run(
	pfl_model=pfl_model,
    dataset_name=dataset_name,
	cohort_size=cohort_size,
	local_iterations=local_iteration,
	local_batch_size=local_batch_size,
	learning_rate=learning_rate,
	samples_per_user=samples_per_user,
	central_iterations=central_iterations,
	cnt_iteration=cnt_iteration
)

# Extract the max central val accuracy and save to a json with hyperparameters, and val loss
save_best_stats_per_iteration(
	iteration=cnt_iteration,
	hyperparameters={
		"local_iterations": local_iteration,
		"learning_rate": learning_rate,
		"samples_per_user": samples_per_user,
		"cohort_size": cohort_size,
		"sampling_probability": sampling_probability,
		"central_iterations": central_iterations,
		"population_size": population_size,
		"local_batch_size": local_batch_size,
		"momentum": momentum,
		"weight_decay": weight_decay
	},
	dataset_name=dataset_name,
    base_folder_metrics=base_folder_metrics,
	base_path_metrics=base_path_metrics
)